<a href="https://colab.research.google.com/github/LuciaMellini/AMD_project/blob/main/findingSimilarItems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding similar items

We download the Letterboxd dataset from Kaggle, using a token.

In [1]:
import os
import json
import pandas as pd
import pip
import string
import re
import numpy as np

os.environ['KAGGLE_USERNAME'] = "xxx"
os.environ['KAGGLE_KEY'] = "xxx"

/usr/local/lib/python3.11/dist-packages/_distutils_hack/__init__.py:31: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


In [2]:
#! kaggle datasets download -d gsimonx37/letterboxd

We only consider a subset of the files contained in the `letterboxd` dataset, namely the data regarding the movie names and ids, their actors, crews, genres and themes.

In [3]:
# import zipfile
# from multiprocessing import Pool

DATA_DIR = "./letterboxd"
# members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
# with zipfile.ZipFile(DATA_DIR + ".zip","r") as zip_ref:
#     for file_name in members_to_extract:
#         zip_ref.extract(file_name + '.csv', DATA_DIR)
!tar xf ./drive/MyDrive/letterboxd.tar.gz

We then prepare the entry point for the Spark functionalities that will we use from now on.

In [6]:
!apt-get update -q
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
#!wget https://downloads.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
!tar xf ./drive/MyDrive/spark-3.5.3-bin-hadoop3.tgz
!pip install -q findspark

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,606 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,230 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/

In [7]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = "./drive/MyDrive/spark-3.5.3-bin-hadoop3"

import findspark
findspark.init("spark-3.5.3-bin-hadoop3")
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.executor.memory", "4g") \
    .appName("ColabSpark") \
    .getOrCreate()

sc = spark.sparkContext

We begin by getting the input files into RDD form. We bring all the strings  to lower case to facilitate later manipulation.

In [8]:
import csv
from io import StringIO

def csv_to_rdd(filename):
    raw = sc.textFile(filename).map(lambda s: s.lower())
    def parse_csv(line):
        reader = csv.reader(StringIO(line))
        return next(reader)
    return raw.map(parse_csv)

def get_column_names(rdd):
    column_names = rdd.filter(lambda r: r[0]=='id').collect()[0][1:]
    return column_names

def prepare_data(filename):
    rdd = csv_to_rdd(filename).cache()
    column_names = get_column_names(rdd)
    result = (rdd
            .filter(lambda r: r[0]!='id')
            .map(lambda r: (int(r[0]), dict(zip(column_names, r[1:])))))
    return result


In [9]:
members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
letterboxd_RDDs={}

for member in members_to_extract:
    letterboxd_RDDs[member] = prepare_data(os.path.join(DATA_DIR, f"{member}.csv"))

A glimpse at the structure of the rows in the RDDs of each member.

In [10]:
for member in members_to_extract:
    print(f"Row for {member:}:\t {letterboxd_RDDs[member].first()}")

Row for actors:	 (1000001, {'name': 'margot robbie', 'role': 'barbie'})
Row for crew:	 (1000001, {'role': 'director', 'name': 'greta gerwig'})
Row for genres:	 (1000001, {'genre': 'comedy'})
Row for movies:	 (1000001, {'name': 'barbie', 'date': '2023', 'tagline': "she's everything. he's just ken.", 'description': 'barbie and ken are having the time of their lives in the colorful and seemingly perfect world of barbie land. however, when they get a chance to go to the real world, they soon discover the joys and perils of living among humans.', 'minute': '114', 'rating': '3.86'})
Row for themes:	 (1000001, {'theme': 'humanity and the world around us'})


Below we have prepared a function to extract a sample of the data, reliant on the ids in the datasets. The maximum size of the sample is $125641$.


In [11]:
ID_MIN = 1000000
SAMPLE = True

def get_sample(rdd, size):
    """ Extract a sample of records from the RDD based on a specified size

    Args:
        rdd (pyspark.RDD): The input RDD containing records, where each record's first element is expected
                           to be an ID as a string.
        size (int): The desired number of records to sample. The function filters records with IDs less
                    than or equal to 1,000,000 plus the specified size.

    Returns:
        pyspark.RDD: An RDD containing the filtered sample of records.
    """
    return rdd.filter(lambda r: r[0]<= ID_MIN+size)

In [12]:
if SAMPLE:
    sample_size = 100
    sample_size_bc = sc.broadcast(sample_size)
    letterboxd_RDDs_sample = {}
    for member in members_to_extract:
        letterboxd_RDDs[member] = get_sample(letterboxd_RDDs[member], sample_size_bc.value)

For each member the available attributes are the following:

| **Member**    | **Attributes**                             |
|--------------|-----------------------------------------|
| **actor**    | name, role                          |
| **crew**     | role, name                          |
| **genres**   | genres                               |
| **movies**   | name, date, tagline, description, minute, rating |
| **themes**   | themes                               |
</br>

For this project we would like to focus on the following features:
<a name="table1"></a>

| **Member**    | **Attributes**                             |
|--------------|-----------------------------------------|
| **actor**    | names of the first 6 actors for a given movie                      |
| **crew**     | name(s) of the director of each movie                        |
| **genres**   | genres                               |
| **movies**   | name, date, minute, rating |
| **themes**   | themes                               |


We set up some primitives to manipulate the values in the RDDs' rows, that are of type `dict`.

In [13]:
from functools import reduce, partial

ensure_list = lambda v: v if isinstance(v, list) else [v]
reduce_dicts = lambda x, y: {key: ensure_list(x[key]) + ensure_list(y[key]) for key in x.keys()}

remove_dict_item = lambda d, key: (d.pop(key), d)[1] if key in d else d

rename_key_in_dict = lambda d, old_key, new_key: remove_dict_item({**d, new_key: d.pop(old_key)} if old_key in d else d, old_key)

update_dict_value = lambda d, key, new_value: {**d, **{key: new_value}} if key in d else d

apply_dict_operations = lambda d, ops: reduce(lambda acc, op: op(acc), ops, d)

For the *crew* member we only keep the directors, and only their name.

In [14]:
operations = [
    partial(rename_key_in_dict, old_key='name', new_key='director'),
    partial(remove_dict_item, key='role')
]

letterboxd_RDDs['crew'] = (letterboxd_RDDs['crew']
                            .filter(lambda r: r[1]['role']=='director')
                            .map(lambda r: (r[0], apply_dict_operations(r[1], operations))))

Some examples of polished *crew* tuples.

In [15]:
letterboxd_RDDs['crew'].take(5)

[(1000001, {'director': 'greta gerwig'}),
 (1000002, {'director': 'bong joon-ho'}),
 (1000003, {'director': 'daniel scheinert'}),
 (1000003, {'director': 'daniel kwan'}),
 (1000004, {'director': 'david fincher'})]

For each category we account for movies having multiple values for a given attribute. In general given rows $$(id_1, \{k_{a}: v_{a_1}, k_{b}: v_{b_1}, \dots, k_{i}: v_{i_1}\dots\})\\(id_1, \{k_{a}: v_{a_2}, k_{b}: v_{b_2}, \dots, k_{i}: v_{i_2}\dots\})$$ the following reduction produces a row  $$(id_1, \{k_{a}: [v_{a_1}, v_{a_2}], k_{b}: [v_{b_1}, v_{b_2}], \dots, k_{i}: [v_{i_1}, v_{i_2}]\dots\})$$

In [16]:
for member in members_to_extract:
    letterboxd_RDDs[member] = letterboxd_RDDs[member].reduceByKey(lambda a, b: reduce_dicts(a, b))

Concerning the information in the *actor* member, we keep only the names of the  most relevant `n_actors` (actors) in each movie. Since the previous transformations on the `letterboxd_RDDs['movies']` (filter, map and reduceByKey) operate in a scanning fashion over the rows, we assume that the actors are still in their original orded in this stage. Letterboxd arranges the actors for a movie by importance in the feature.

In [17]:
n_actors = 6

operations_actors = [
    partial(remove_dict_item, key='role'),
    partial(rename_key_in_dict, old_key='name', new_key='actors'),
    lambda d: update_dict_value(d, 'actors', d['actors'][:n_actors]),
    lambda d: {**d, **{f'actor{i+1}': actor for i, actor in enumerate(d['actors'])}},
    partial(remove_dict_item, key='actors')
]

letterboxd_RDDs['actors'] = (letterboxd_RDDs['actors']
                            .map(lambda r: (r[0], apply_dict_operations(r[1], operations_actors))))

For example,

In [18]:
letterboxd_RDDs['actors'].first()

(1000002,
 {'actor1': 'song kang-ho',
  'actor2': 'lee sun-kyun',
  'actor3': 'cho yeo-jeong',
  'actor4': 'choi woo-shik',
  'actor5': 'park so-dam',
  'actor6': 'lee jung-eun'})

For each *movie* we only store the attributes listed in the <a href="#table1">table above</a>.

In [19]:
operations_movies = [
    partial(remove_dict_item, key='tagline'),
    partial(remove_dict_item, key='description')
]

letterboxd_RDDs['movies'] = (letterboxd_RDDs['movies']
                            .map(lambda r: (r[0], apply_dict_operations(r[1], operations_movies))))

Now that we have polished each member in the dataset, we collect all the features for a given film in a single dictionary. So, a generic row of `movies_RDD` has the following format:
<p align=center><i>(id, {category: value(s)})</i></p>

In [20]:
movies_RDD = letterboxd_RDDs[members_to_extract[0]]
for member in members_to_extract[1:]:
    movies_RDD = movies_RDD.union(letterboxd_RDDs[member])
movies_RDD = movies_RDD.reduceByKey(lambda a, b: {**a, **b})

For example,

In [21]:
id, value = movies_RDD.first()
print("id:\t {}\nfeatures:\t {}".format(id,value))

id:	 1000086
features:	 {'actor1': "lupita nyong'o", 'actor2': 'winston duke', 'actor3': 'shahadi wright joseph', 'actor4': 'evan alex', 'actor5': 'tim heidecker', 'actor6': 'elisabeth moss', 'director': 'jordan peele', 'genre': ['thriller', 'horror'], 'name': 'us', 'date': '2019', 'minute': '116', 'rating': '3.65', 'theme': ['horror, the undead and monster classics', 'intense violence and sexual transgression', 'terrifying, haunted, and supernatural horror', 'twisted dark psychological thriller', 'creepy, chilling, and terrifying horror', 'gothic and eerie haunting horror', 'gory, gruesome, and slasher horror']}


## Data pre-processing

To preserve the independent role of each attribute we have decided to measure their similarity using cosine distance. This entails translating all data regarding a movie into a vector with components in $\mathbb{R}$.

Below we list the data types of the various attributes.

| **Attribute**    | **Datatype**                       |
|--------------|-----------------------------------------|
| **actor**    | string         |
| **director**     | string                        |
| **genre**   | string                             |
| **theme**   | string                             |
| **name**   | string |
| **date**   | numerical |
| **minute**   | numerical |
| **rating**   | numerical                             |

It is evident that the textual attributes have to be transformed into values to be able to work in a Euclidean space. The following paragraphs are dedicated to these transformations. We refer to the report for a discussion regarding the chosen methods.


### String pre-processing

In [22]:
categories_all = [f'actor{i+1}' for i in range(n_actors)]+['director', 'genre', 'name', 'theme', 'date', 'minute', 'rating']

In [23]:
apply_to_list_or_value = lambda func, value: [func(v) for v in value] if isinstance(value, list) else func(value)

In [24]:
movies_processed_RDD = movies_RDD

#### Natural language processing

To distill the semantics of the movie's theme we apply the following natural language processing steps:
* remove stop words
* replace the words with their lemmatized version

In [25]:
import spacy
! python -m spacy download en_core_web_md -q

nlp = spacy.load("en_core_web_md")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 18.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [26]:
remove_punctuation = lambda x: re.sub(r'[^\w\s]','',x)
remove_multiple_spaces = lambda x: re.sub(r'\s+',' ',x)
remove_stop_words_func = lambda x: " ".join([token.text for token in nlp(x) if not token.is_stop])
lemmatize_func = lambda x: " ".join([token.lemma_ for token in nlp(x)])

chain_functions = lambda *funcs: lambda x: reduce(lambda acc, f: f(acc), funcs, x)
nlp_processing = chain_functions(remove_punctuation, remove_multiple_spaces, remove_stop_words_func, lemmatize_func)

categories_nlp = ['theme']

operations_nlp = [lambda d, cat=cat: update_dict_value(d, cat, apply_to_list_or_value(nlp_processing, d[cat])) for cat in categories_nlp]

In [27]:
movies_processed_RDD = movies_processed_RDD.map(lambda r: (r[0], apply_dict_operations(r[1], operations_nlp)))

#### Embedding strings

We use the spaCy text vectorizations of the attributes:
* name
* genre
* theme

In [28]:
embedding_func = lambda x: nlp(x).vector

categories_vec = ['name', 'genre', 'theme']

operations_vec = [lambda d, cat=cat: update_dict_value(d, cat, apply_to_list_or_value(embedding_func, d[cat])) for cat in categories_vec]

In [29]:
movies_processed_RDD = movies_processed_RDD.map(lambda r: (r[0], apply_dict_operations(r[1], operations_vec)))

For example a value for the *name* attribute will result as such:

In [30]:
id, value = movies_processed_RDD.first()
print("id:\t {}\nvalue:\t {}".format(id,value['name']))

id:	 1000063
value:	 [-3.7368002   1.228715   -3.57195     2.304045    2.79185     0.72595
  2.75385     5.9788      0.654925    1.0706955   3.7602298   0.01811001
 -3.1328502   0.4428344   0.49480498  0.16201001 -1.2791901   1.76125
  2.08552     2.5367     -1.7260349   2.6556      2.3859     -5.1029
  2.6268     -0.25960004 -1.97415    -2.46988     0.565505   -1.80498
 -2.346795    2.1229     -0.84890497 -1.1215025  -2.53675    -1.2255399
  0.14168    -0.03965998 -0.898705   -2.9952502   0.33808348 -1.52151
 -0.894325    1.9627      1.4762      2.90875    -2.08515    -1.68365
  0.1325      3.870265   -1.8426905   3.2546     -0.17685002  0.21201
 -0.92254305  0.3136      2.2937198  -0.25241497  1.078035    0.19895
  0.718083   -2.22015    -2.0486      0.0592335   0.66233    -0.612065
 -2.3470001  -4.56115     2.60475     2.435595    1.11834    -1.48194
 -1.138085    2.1680999  -0.34965748  0.43174002 -1.6201      1.6717
 -0.63365996  0.21365    -1.7428501  -0.302035   -0.94395     2.7

Since for each movie there are multiple genres and themes, for these attributes the dictionary contains a list of numpy arrays.

#### Hashing strings

We hash the strings of the features:
* actors
* director

In [31]:
import math
import hashlib

def hash_object(byte_obj, hash_bucket_size, salt=bytes(0)):
    """ Hash an object into a bucket of values [0,hash_bucket_size-1] on the basis of the seed

        Args:
            byte_obj (byte array): an object in byte format
            salt (byte array): salt for the hash function
            hash_bucket_size (int): the size of the bucket to which the objects get hashed to

        Returns:
            int: hashed object
        """
    m=hashlib.shake_256()
    m.update(salt)
    m.update(byte_obj)
    required_bytes = math.ceil(hash_bucket_size / 8)
    hash_output = m.digest(required_bytes)
    hashed_value = int.from_bytes(hash_output, 'little')
    return hashed_value % hash_bucket_size

In [32]:
hash_func = lambda v: hash_object(bytes(v,'utf-8'),2**30)

categories_hash = [f"actor{i+1}" for i in range(n_actors)]+['director']

operations_hash = [lambda d, cat=cat: update_dict_value(d, cat, apply_to_list_or_value(hash_func, d[cat])) for cat in categories_hash]

In [33]:
movies_processed_RDD = movies_processed_RDD.map(lambda r: (r[0], apply_dict_operations(r[1], operations_hash)))

For example a value for the *director* attribute will result as such:

In [34]:
id, value = movies_processed_RDD.first()
print("id:\t {}\nvalue:\t {}".format(id,value['director']))

id:	 1000086
value:	 122639349


In [35]:
operations_preprocessing = sum([operations_nlp, operations_vec, operations_hash], [])

### Vector preparation

Now that we have prepared all categories in a targeted way, we can proceed by building the vectors of the movies.

#### Reduce attributes with multiple values

We simply average the values or arrays for attributes that have multiple values. e.g directors, themes, genres.

In [36]:
# cannot assume that the value is a list since e.g. "director" attribute could have a singular value
def average_list_values(v):
    if isinstance(v, list):
        if isinstance(v[0],np.ndarray):
            return np.mean(np.array(v), axis=0)
        return np.mean(v)
    return v

In [37]:
operations_average = [lambda d, cat=cat: update_dict_value(d, cat, average_list_values(d[cat])) for cat in categories_all]

In [38]:
vectors_RDD = movies_processed_RDD.map(lambda r: (r[0], apply_dict_operations(r[1], operations_average)))

In [39]:
def dictionary_to_array_of_values(d):
    values = [v if isinstance(v, np.ndarray) else np.array([v]) for key, v in sorted(d.items())]
    value_array = np.concatenate(values)
    return np.vectorize(lambda x: float(x))(value_array)

In [40]:
vectors_RDD = vectors_RDD.map(lambda r: (r[0], dictionary_to_array_of_values(r[1]))).cache()

For each movie we have a vector of the following dimension:

In [41]:
dim_vectors = len(vectors_RDD.first()[1])
print(f"Each vector has {dim_vectors} dimensions")

Each vector has 910 dimensions


#### Standardization

We standardize the vector's components such that each feature has a sample mean of $0$ and a sample standard deviation of $1$.

In [42]:
ensure_dict = lambda v: v if isinstance(v, dict) else {v[0]:v[1]}

collect_as_map = lambda a, b: {**ensure_dict(a), **ensure_dict(b)}

def get_smean_per_list_index(rdd):
    rdd = (rdd
           .flatMap(lambda r: [(i,(item, 1)) for i,item in enumerate(r[1])])
           .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) )
    rdd_mean = rdd.map(lambda r: (r[0], r[1][0]/r[1][1]))
    return rdd_mean.reduce(collect_as_map)

def get_ssd_per_list_index(rdd, smean_dict):
    rdd_sum_sq_diff_count = (rdd
            .flatMap(lambda r: [(i,((item - smean_dict[i])**2, 1)) for i,item in enumerate(r[1])])
            .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])))
    rdd_ssd = rdd_sum_sq_diff_count.map(lambda r: (r[0], math.sqrt(r[1][0]/r[1][1])))
    return rdd_ssd.reduce(collect_as_map)

In [43]:
def RDD_standardize(vectors_rdd):
    mean_dict = get_smean_per_list_index(vectors_rdd)
    mean_dict_bc = sc.broadcast(mean_dict)
    ssd_dict = get_ssd_per_list_index(vectors_rdd, mean_dict_bc.value)
    ssd_dict_bc = sc.broadcast(ssd_dict)
    return vectors_rdd.map(lambda r:  (r[0], (r[1] - np.array([mean_dict_bc.value[i] for i in range(len(r[1]))])) / np.array([ssd_dict_bc.value[i] for i in range(len(r[1]))])))

In [44]:
vectors_stand_RDD = RDD_standardize(vectors_RDD)

#### Principal-Component Analysis (PCA)

We reduce the amount of features for each vector to avoid suffering from the curse of dimensionality when evaluating their similarity.

We begin by building the covariance matrix for the data, and we compute its eigenvalues and eigenvectors.


In [45]:
def PCA(vectors_rdd):
    """
    Perform Principal Component Analysis (PCA) on an RDD of vectors.

    Args:
        vectors_rdd (pyspark.RDD): An RDD where each element is a tuple with an identifier, and the second part is a vector of numerical features.

    Returns:
        tuple: A tuple containing:
            - sorted_eigenvalues (numpy.ndarray): The eigenvalues sorted in non increasing order.
            - sorted_eigenvectors (numpy.ndarray): The eigenvectors corresponding to the sorted eigenvalues.
    """
    n_vectors = vectors_rdd.count()
    cov_matrix = (vectors_rdd
                    .map(lambda r: (r[0], np.outer(r[1], r[1])))
                    .reduce(lambda a, b: (1, a[1] + b[1])))[1] / n_vectors

    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

    # sort eigenvalues and eigenvectors in non increasing order
    sorted_indices = np.argsort(eigenvalues)[::-1]
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices]
    return (sorted_eigenvalues, sorted_eigenvectors)

In [46]:
sorted_eigenvalues, sorted_eigenvectors = PCA(vectors_stand_RDD)

We only retain the components such that their total cumulative explained variance is at least $95\%$.

In [47]:
def k_principal_components(sorted_eigenvalues, sorted_eigenvectors, t_PCA):
    """
    Select the top-k principal components based on the cumulative explained variance threshold.

    Args:
        sorted_eigenvalues (numpy.ndarray): The eigenvalues sorted in non increasing order.
        sorted_eigenvectors (numpy.ndarray): The eigenvectors corresponding to the sorted eigenvalues.
        t_PCA (float): The cumulative explained variance threshold (a value between 0 and 1).

    Returns:
        numpy.ndarray: The top-k eigenvectors that explain at least the specified threshold of the variance.
    """
    explained_variance = sorted_eigenvalues / np.sum(sorted_eigenvalues)
    cumulative_explained_variance = np.cumsum(explained_variance)
    k = np.argmax(cumulative_explained_variance >= t_PCA) + 1
    top_k_eigenvectors = sorted_eigenvectors[:, :k]
    return top_k_eigenvectors

In [48]:
t_PCA = 0.95
top_k_eigenvectors = k_principal_components(sorted_eigenvalues, sorted_eigenvectors, t_PCA)
k = top_k_eigenvectors.shape[1]

print(f"Number of components explaining at least {t_PCA*100}% of the variance: {k}")

Number of components explaining at least 95.0% of the variance: 53


In [49]:
top_k_eigenvectors_bc = sc.broadcast(top_k_eigenvectors)
vectors_reduced_RDD = vectors_stand_RDD.map(lambda r: (r[0], np.dot(top_k_eigenvectors_bc.value.T, r[1])))

An example of reduced vector:

In [50]:
id, vector = vectors_reduced_RDD.first()
print("movie id:\t {}\n  vector:\t {}".format(id,vector))

movie id:	 1000040
  vector:	 [25.45678573+0.j  0.12065804+0.j -2.63756383+0.j -1.42343519+0.j
  5.08507867+0.j -3.95027141+0.j  0.8219175 +0.j  3.2039606 +0.j
  0.13246169+0.j  2.9730251 +0.j  1.06559536+0.j -0.93789517+0.j
  0.23405023+0.j -1.91881201+0.j  0.54194437+0.j -1.44506389+0.j
  3.81372045+0.j -0.32028173+0.j -0.2941037 +0.j  2.31112813+0.j
  0.93754253+0.j  1.14918825+0.j  1.27860049+0.j  1.08220835+0.j
 -1.56758995+0.j -2.54590296+0.j -1.45645182+0.j  1.49585892+0.j
  1.70019421+0.j -2.57253945+0.j  0.9631534 +0.j -0.55967854+0.j
  1.05008809+0.j  0.68704178+0.j  0.48528741+0.j -0.99323773+0.j
  2.20962485+0.j  1.5121077 +0.j -0.14521523+0.j -0.2523951 +0.j
  0.0701358 +0.j  1.65900701+0.j  2.18206555+0.j  1.075918  +0.j
 -0.83334205+0.j  4.14620218+0.j -0.08746862+0.j  1.99576198+0.j
 -2.37028838+0.j -1.23256378+0.j  0.05430903+0.j -0.67606319+0.j
  0.64904429+0.j]


## Locality Sensitive Hashing (LSH)

### Locality sentitive family for cosine distance

We build the locality sensitive family $\mathcal{F}$ as a set of randomly chosen vectors $\{v_{f\in\mathcal{F}}\}$. Given two vectors $x$ and $y$, they make a candidate pair of similar items if and only if the dot products $x\cdot v_f$ and $x \cdot v_f$ have the same sign. A family of functions $\mathcal{F}$ built as described is a locality-sensitive family for the cosine distance.

We will also refer to the random vectors in $\mathcal{F}$ as hash functions.

Since we will compute the dot product between each of the elements in the dataset and all of the hash functions in $\mathcal{F}$, we try to simplify the computation of the cosine distance between vectors. We do so by restricting the random choice of vectors to those having components $+1$ or $-1$. Hence the dot product of any vector $x$ with a vector in such a family $\mathcal{F}$ is given by its algebraic sum $x$'s components, where the signs depend on the components of the random vector.

In [70]:
def RDD_LS_hash_family_cosine_distance(signature_length, num_components):
    """
    Generate a locality sentitive family for cosine distance.

    Args:
        signature_length (int): The signature length for the items.
        num_components (int): he number of components for the random vectors.
    Returns:
        pyspark.RDD: An RDD of hash functions, where each hash function is a tuple with an index and a random vector in {-1,1}^k.
    """
    # pass through np.array for reproducibility reasons
    np.random.seed(0)
    hash_funcs = np.random.choice([1, -1], size=(signature_length, num_components))
    hash_funcs_rdd = sc.parallelize(list(enumerate(hash_funcs)))

    return hash_funcs_rdd

For each of the (reduced) vectors in the dataset we build its signature by computing its dot product with all vectors in $\mathcal{F}$ (called `hash_funcs_RDD` in the code).

In [71]:
def RDD_signatures(vectors_rdd, hash_funcs_rdd, k):
    """
    Generate locality-sensitive hashing (LSH) signatures for a set of vectors.

    Args:
        vectors_rdd (pyspark.RDD): An RDD where each element is a tuple. The first part is an identifier, and the second part is a vector of numerical features.
        signature_length (int): The signature length for the items.
    Returns:
        pyspark.RDD: An RDD where each element is a tuple containing an identifier and its corresponding LSH signature.
    """
    hash_funcs_rdd = RDD_LS_hash_family_cosine_distance(signature_length, k)

    signatures_rdd = (vectors_rdd.cartesian(hash_funcs_rdd)
                    .map(lambda r: ((r[0][0],r[1][0]), (r[0][1], r[1][1])))
                    .map(lambda r: (r[0][0], (r[0][1], np.sign(np.dot(r[1][0], r[1][1])))))
                    .groupByKey().mapValues(list)
                    .map(lambda r: (r[0], np.array([int(v[1]) for v in sorted(r[1])]))))                            # numpy uses complex numbers
    return signatures_rdd

In [187]:
signature_length=200
signatures_RDD = RDD_signatures(vectors_reduced_RDD, signature_length, k)

Let's give look at a possibile signature for a movie.

In [188]:
id, signature = signatures_RDD.filter(lambda r: r[0]==1000040).first()
print(" movie id:\t {}\nsignature:\t {}".format(id,signature))

 movie id:	 1000040
signature:	 [ 1 -1  1 -1  1 -1 -1  1 -1 -1  1  1  1 -1  1  1  1 -1 -1  1  1 -1  1 -1
  1  1  1  1 -1  1 -1  1 -1  1 -1 -1  1 -1  1 -1 -1  1  1  1  1  1  1  1
 -1 -1 -1  1  1  1  1 -1  1  1 -1  1  1  1  1  1 -1 -1 -1  1  1 -1  1  1
 -1  1  1 -1  1  1  1  1 -1 -1 -1 -1 -1  1 -1  1  1  1 -1  1  1  1 -1 -1
 -1 -1 -1 -1  1  1  1  1 -1  1 -1  1  1  1 -1 -1  1 -1  1 -1  1  1  1 -1
  1  1 -1  1 -1  1  1  1  1 -1  1 -1 -1  1 -1  1  1 -1  1  1  1 -1  1 -1
  1 -1  1 -1  1  1 -1 -1 -1  1  1  1  1 -1  1 -1  1 -1 -1  1 -1  1 -1 -1
  1 -1 -1  1 -1  1 -1  1  1  1 -1  1 -1  1  1  1 -1 -1 -1  1  1 -1  1 -1
 -1 -1  1  1 -1  1 -1  1]


Now, `signatures_RDD` contains the so called *signature matrix*, that contains a signature for each movie in the dataset.

### Banding technique

It would be unthinkable to compare all possible pairs of movies to find similar ones among them. This would mean scanning all the rows in the signature matrix to compute the relative frequency between possible pairs of movies. So, we proceed by applying locality-sensitive hashing. In this approach we reduce the number of rows that determine the signature of a movie by hashing so called bands of rows. The rationale is that similar movies are more likely to be hashed in the same bucket, so we hope that dissimilar pairs end up in distinct buckets, and thus are never checked for similarity. Looking at the resulting signatures we consider as a candidate pair only those for which their cosine similarity exceeds a threshold $t$.

We begin by dividing the signature matrix into $b$ bands of $r$ rows each. The choice of $r$ and $b$ depends on the threshold $t$ on the cosine distance between pairs of movies. The value of the threshold $t$ is approximately the value of similarity at which the probability of becoming a candidate is $\frac{1}{2}$.
So, we keep into account the following relationships (for brevity $l$=`signature_length`):   

\begin{equation*}
    \begin{cases}
        t=\left(\frac{1}{b}\right)^\frac{1}{r} \\
        r\cdot b = l
    \end{cases}
\end{equation*}

The function `band_size` solves this system in $r$ by computing it's value through
\begin{equation*}
    r=-\frac{W(-l\ln(t))}{\ln(t)}
\end{equation*}

where $W$ is the Lambert $W$ function, used to solve equations in the form $we^{w}=z$ for $w$.

In [189]:
from scipy.special import lambertw
import math

def band_size(t, signature_length):
    """
    Compute the number of rows that form a band for the LSH technique.

    Args:
        t (int): The desired threshold in [0,1] on the similarity between pairs of items.
        signature_length (int): The signature length for the items.

    Returns:
        int: The ideal number of rows contained in the band.
    """
    return -math.ceil(lambertw(-signature_length*np.log(t)).real/np.log(t))

def r_b_choice(t,signature_length):
    """
    Choose the adeguate number of rows a band and the number of bands for the LSH technique.

    Args:
        t (int): The desired threshold in [0,1] on the similarity between pairs of items.
        signature_length (int): The signature length for the items.

    Returns:
        int: The ideal number of rows, and consequent number of bands on the basis of the signature length.
    """
    r=band_size(t,signature_length)
    b=math.ceil(signature_length/r)
    return (r,b)

In [200]:
t=0.9
t_bc = sc.broadcast(t)
r,b = r_b_choice(0.8,signature_length)
b_bc = sc.broadcast(b)
print("The chosen parameters are: \n r: {} \n b: {}".format(r,b))

The chosen parameters are: 
 r: 12 
 b: 17


Having chosen the parameters we proceed by subdividing the rows of the similarity matrix into bands.

In [201]:
split_list = lambda l, b: np.array_split(np.array(l), b)

We proceed by hashing the rows in each of the $b$ bands for each vector. For each band we use a different bucket array, so that signatures with two equal vectors in separate bands are hashed differently. This is done by adding some salt to the hash function with depending on the band.

In [202]:
def RDD_banding(signatures_rdd, b, hash_bucket_size):
    """ Compute the hashed signatures with the LSH technique

    Args:
        signatures_rdd (RDD): RDD of (set key, signature for set)
        b (int): number of bands in which to split the signature matrix represented by signatures_rdd
        hash_bucket_size (int): the size of the bucket to which the signature portions in each band get hashed to

    Returns:

    """
    b_bc = sc.broadcast(b)
    hash_bucket_size_bc = sc.broadcast(hash_bucket_size)

    banding_func = chain_functions(partial(split_list, b=b_bc.value), lambda l: [(hash_object(bytes(str(l),'ascii'),hash_bucket_size_bc.value,bytes(i))) for i in range(b)])

    return signatures_rdd.map(lambda r: (r[0],banding_func(r[1])))

Also, to avoid hashing distinct portions of a signature in the same bucket it is important to choose a great enough bucket. Here we have evaluted the number of tuples given by $\{-1,1\}^r$.

In [203]:
hash_bucket_size = 2**(r+2)

hashed_signatures_RDD = RDD_banding(signatures_RDD, b, hash_bucket_size)

Let's give a look at the new compact representation of a movie.

In [204]:
id, signature = hashed_signatures_RDD.first()
print(" movie id:\t {}\nsignature:\t {}".format(id,signature))

 movie id:	 1000040
signature:	 [3494, 9506, 11764, 7430, 1432, 296, 6810, 10583, 11755, 16327, 13026, 6376, 6902, 13107, 8822, 8050, 2741]


### Find similar items

Now we search for candidate pairs among the reviews. We consider as possible similar couples of reviews those that have cosine similarity at least $t$.

In [205]:
import itertools

def RDD_candidate_pairs(signature_rdd):
    result = (signature_rdd.flatMap(lambda r: [((i, el),r[0]) for i, el in enumerate(r[1])])
                        .groupByKey()
                        .filter(lambda r: len(r[1])>1)
                        .flatMap(lambda r: itertools.combinations(r[1], 2))
                        .map(lambda r: (r[0],r[1]) if r[0]<r[1] else (r[1],r[0]))
                        .reduceByKey(lambda a,b: a))
    return result

keep_values_with_common_elements = lambda r: any(x == y for x, y in zip(r[1][0], r[1][1]))

cosine_distance = lambda x,y: np.arccos(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))

cosine_similarity = lambda x,y: (math.pi-cosine_distance(x,y))/math.pi

similarity_over_threshold = lambda x, y, sim_func, t: sim_func(x,y)>=t

def RDD_candidate_pairs_with_vectors(vectors_rdd, candidate_pairs_rdd):
    candidate_pairs_with_vectors_rdd = (candidate_pairs_rdd
                                        .leftOuterJoin(vectors_rdd)
                                        .map(lambda r: (r[1][0],(r[0],r[1][1])))
                                        .leftOuterJoin(vectors_rdd)
                                        .map(lambda r: ((r[1][0][0], r[0]), (r[1][0][1], r[1][1]))))
    return candidate_pairs_with_vectors_rdd

In [206]:
candidate_pairs_RDD = RDD_candidate_pairs(hashed_signatures_RDD).cache()
candidate_pairs_with_vectors_RDD = RDD_candidate_pairs_with_vectors(vectors_RDD, candidate_pairs_RDD)
similar_items_with_similarity_RDD = (candidate_pairs_with_vectors_RDD
                                    .filter(lambda r: similarity_over_threshold(r[1][0], r[1][1], cosine_similarity, t_bc.value))
                                    .map(lambda r: (r[0],cosine_similarity(r[1][0],r[1][1]))))

In [207]:
similar_items_with_similarity_RDD = similar_items_with_similarity_RDD.coalesce(1)

In [208]:
parameters = {
    "t_PCA": t_PCA,
    "l": signature_length,
    "t": t
}
r,b = r_b_choice(0.8, parameters["l"])
parameters = {**parameters, **{"b": b, "h":hash_bucket_size}}

experiment_n = 12
with open(f"parameters_experiment_{experiment_n}.json", "w") as f:
    json.dump(parameters, f)

similar_items_with_similarity_RDD.saveAsTextFile(f"similar_items_experiment_{experiment_n}.txt")

## Experiments

In this paragraph we predispose the code to run an experiment. We have followed the outline described in Section 3.2.2 of the report.

In [121]:
# def data_preprocessing(data_rdd, t_PCA):
#     # Data embedding
#     data_processed_rdd = data_rdd.map(lambda r: (r[0], apply_dict_operations(r[1], operations_preprocessing)))
#     data_processed_rdd = data_processed_rdd.map(lambda r: (r[0], apply_dict_operations(r[1], operations_average)))
#     vectors_rdd = data_processed_rdd.map(lambda r: (r[0], dictionary_to_array_of_values(r[1]))).cache()

#     # Standardize the vector
#     vectors_stand_rdd = RDD_standardize(vectors_rdd)

#     # Apply PCA, keeping k components
#     sorted_eigenvalues, sorted_eigenvectors = PCA(vectors_stand_rdd)
#     top_k_eigenvectors = k_principal_components(sorted_eigenvalues, sorted_eigenvectors, t_PCA)
#     k = top_k_eigenvectors.shape[1]
#     top_k_eigenvectors_bc = sc.broadcast(top_k_eigenvectors)
#     vectors_reduced_rdd = vectors_stand_rdd.map(lambda r: (r[0], np.dot(top_k_eigenvectors_bc.value.T, r[1])))
#     return vectors_reduced_RDD, k

# def find_similar_items(vectors_rdd, k, l, t, b, h):
#     # Build signature matrix
#     signatures_rdd = RDD_signatures(vectors_RDD, l, k).cache()

#     # Divide signature matrix into bands and hash the bands
#     hashed_signatures_rdd = RDD_banding(signatures_rdd, b, h).cache()

#     # Determinet candidate pairs
#     candidate_pairs_rdd = RDD_candidate_pairs(hashed_signatures_rdd).cache()

#     # Filter actual similar items
#     candidate_pairs_with_vectors_rdd = RDD_candidate_pairs_with_vectors(vectors_rdd, candidate_pairs_rdd)
#     similar_items_with_similarity_rdd = candidate_pairs_with_vectors_rdd.filter(lambda r: similarity_over_threshold(r[1][0], r[1][1], cosine_similarity, t_bc.value))
#     return similar_items_with_similarity_rdd

# def run_experiment(data_rdd, t_PCA, l, t, b, h):
#     vectors_rdd, k = data_preprocessing(data_rdd, t_PCA)
#     print(f"Number of components explaining at least {t_PCA*100}% of the variance: {k}")
#     similar_items_with_similarity_rdd = find_similar_items(vectors_rdd, k, l, t, b, h)
#     return similar_items_with_similarity_rdd

In [122]:
experiment_n = 0

parameters = {
    "t_PCA": t_PCA,
    "l": signature_length,
    "t": t
}
r,b = r_b_choice(0.8, parameters["l"])
parameters = {**parameters, **{"b": b, "h":hash_bucket_size}}

with open(f"parameters_experiment_{experiment_n}.json", "w") as f:
    json.dump(parameters, f)

# result = run_experiment(movies_RDD, *parameters.values()).coalesce()
# result.saveAsTextFile(f"similar_items_experiment_{experiment_n}.txt")